# Research Assistant — MCP + Gemini Agentic Application
**Mini-Project | DI Bootcamp**

## Theme: Research Assistant
An end-to-end agentic workflow that:
1. Reads and writes files via **@modelcontextprotocol/server-filesystem** (MCP)
2. Tracks work via **@modelcontextprotocol/server-memory** (MCP)
3. Runs custom research tools via a **FastMCP custom server** (citation formatter, keyword extractor, markdown report generator)
4. Uses **Gemini** as the LLM orchestrator via LangGraph

The agent decides which tools to call — no hard-coded flow.


---
## Step 1 — Install Dependencies


In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio"


---
## Step 2 — Verify Node/NPM (required for MCP servers)


In [ ]:
import subprocess
result = subprocess.run(["node", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("Node not found — installing...")
    import os
    os.system("apt-get -qq update && apt-get -qq install -y nodejs npm")
else:
    print("Node:", result.stdout.strip())

result2 = subprocess.run(["npx", "--version"], capture_output=True, text=True)
print("NPX:", result2.stdout.strip())


---
## Step 3 — Set Google API Key


In [ ]:
import os
from google.colab import userdata

# Option A: from Colab secrets (recommended)
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets")
except Exception:
    # Option B: paste directly (not recommended for shared notebooks)
    os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
    print("API key set manually")


---
## Step 4 — Create Working Directory & Sample Research Files


In [ ]:
from pathlib import Path

WORKDIR = "/content/research_workspace"
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

# Sample research papers (text summaries)
papers = {
    "paper_attention.txt": (
        "Title: Attention Is All You Need\n"
        "Authors: Vaswani et al., 2017\n"
        "Journal: NeurIPS\n"
        "Abstract: We propose a new simple network architecture, the Transformer, "
        "based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. "
        "Experiments on two machine translation tasks show these models to be superior in quality "
        "while being more parallelizable and requiring significantly less time to train.\n"
        "Keywords: transformer, attention mechanism, NLP, translation, deep learning\n"
    ),
    "paper_bert.txt": (
        "Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding\n"
        "Authors: Devlin et al., 2019\n"
        "Journal: NAACL\n"
        "Abstract: We introduce BERT, a new language representation model, which stands for "
        "Bidirectional Encoder Representations from Transformers. Unlike recent language representation "
        "models, BERT is designed to pre-train deep bidirectional representations from unlabeled text "
        "by jointly conditioning on both left and right context in all layers.\n"
        "Keywords: BERT, pre-training, bidirectional, language model, NLP, fine-tuning\n"
    ),
    "paper_gpt.txt": (
        "Title: Language Models are Few-Shot Learners (GPT-3)\n"
        "Authors: Brown et al., 2020\n"
        "Journal: NeurIPS\n"
        "Abstract: We show that scaling up language models greatly improves task-agnostic, few-shot "
        "performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning "
        "approaches. GPT-3 has 175 billion parameters and can perform tasks with few examples.\n"
        "Keywords: GPT-3, few-shot learning, large language model, scaling, NLP\n"
    ),
    "references.txt": (
        "Raw references to format:\n"
        "1. vaswani 2017 attention all you need neurips\n"
        "2. devlin 2019 bert naacl pre-training transformers\n"
        "3. brown 2020 gpt3 few-shot neurips\n"
    ),
}

for fname, content in papers.items():
    (Path(WORKDIR) / fname).write_text(content)

print("Workspace created:", WORKDIR)
print("Files:", [f.name for f in Path(WORKDIR).iterdir()])


---
## Step 5 — Custom FastMCP Server (Research Tools)

This server exposes 4 research-specific tools:
- `ping()` — health check
- `extract_keywords(text)` — extract and count unique keywords from paper text
- `format_citation(raw)` — format a raw reference string into APA style
- `build_summary_report(papers_data)` — generate a structured Markdown report


In [ ]:
import textwrap
from pathlib import Path

server_path = Path("/content/research_mcp_server.py")
server_path.write_text(textwrap.dedent('''
    from fastmcp import FastMCP
    from typing import Dict, List
    import re

    mcp = FastMCP(name="research_tools")

    @mcp.tool
    def ping() -> str:
        """Health check."""
        return "pong — research_tools server is alive"

    @mcp.tool
    def extract_keywords(text: str) -> Dict[str, object]:
        """Extract unique keywords from a research paper text.
        Looks for a Keywords: line and also frequent capitalized terms.
        Returns a dict with keyword_list and count.
        """
        keywords = []
        for line in text.splitlines():
            if line.lower().startswith("keywords:"):
                raw = line.split(":", 1)[1]
                keywords = [k.strip() for k in raw.split(",") if k.strip()]
                break
        if not keywords:
            words = re.findall(r'\b[A-Z][a-z]{3,}\b', text)
            freq = {}
            for w in words:
                freq[w] = freq.get(w, 0) + 1
            keywords = [w for w, c in sorted(freq.items(), key=lambda x: -x[1]) if c > 1][:8]
        return {"keyword_list": keywords, "count": len(keywords)}

    @mcp.tool
    def format_citation(author: str, year: str, title: str,
                        journal: str, doi: str = "") -> str:
        """Format a citation in APA style.
        Returns a clean APA-formatted reference string.
        """
        apa = f"{author} ({year}). {title}. {journal}."
        if doi:
            apa += f" https://doi.org/{doi}"
        return apa

    @mcp.tool
    def build_summary_report(topic: str, papers: List[str],
                             keywords: List[str]) -> str:
        """Generate a structured Markdown literature review report.
        Args:
            topic: Research topic title
            papers: List of paper titles included
            keywords: List of key themes extracted
        Returns a Markdown-formatted report string.
        """
        paper_list = chr(10).join(f"- {p}" for p in papers)
        kw_list = ", ".join(f"`{k}`" for k in keywords)
        report = f"""# Literature Review: {topic}

## Papers Reviewed
{paper_list}

## Key Themes
{kw_list}

## Summary
This review covers {len(papers)} papers on the topic of {topic}.
The recurring themes across the literature include: {kw_list}.

## Next Steps
- Cross-reference citations
- Identify research gaps
- Synthesize findings into a narrative review
"""
        return report

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''), encoding="utf-8")

print("Custom MCP server written to:", server_path)


---
## Step 6 — Connect to MCP Servers

We register 3 servers:
1. **filesystem** — read/write files in the workspace
2. **memory** — persistent key-value memory across agent turns
3. **research_tools** — our custom FastMCP server


In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from langchain_mcp_adapters.client import MultiServerMCPClient

server_path = Path("/content/research_mcp_server.py")

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "memory": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-memory"],
    },
    "research_tools": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print(f"Total tools available: {len(tools)}")
print("\nAll tools:")
for t in tools:
    print(f"  [{t.name.split('_')[0]:15}]  {t.name}")


---
## Step 7 — Build the Gemini Agent with LangGraph

The agent uses a **ReAct loop**: Gemini decides which tool to call, observes the result, and decides the next step — no hard-coded flow.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.2,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

system_prompt = """
You are a Research Assistant agent with access to filesystem, memory, and research tools.

Your capabilities:
- READ files from the research workspace using filesystem tools
- EXTRACT keywords from paper text using research_tools__extract_keywords
- FORMAT citations in APA style using research_tools__format_citation
- BUILD markdown reports using research_tools__build_summary_report
- WRITE reports back to the filesystem using filesystem tools
- STORE notes in memory using memory tools

Always:
1. Start by reading the relevant files
2. Use the appropriate research tools to process them
3. Write results back to the workspace
4. Store key findings in memory for future reference
"""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("Agent created with Gemini 2.0 Flash")


---
## Step 8 — Task 1: Extract Keywords from All Papers

The agent reads each paper file and extracts its keywords autonomously.


In [ ]:
task1 = """
Read all the .txt files in the workspace that start with 'paper_'.
For each paper, extract its keywords using the extract_keywords tool.
Then store a summary of the keywords found in memory under the key 'paper_keywords'.
Report back a summary of all keywords found across all papers.
"""

print("Running Task 1: Keyword extraction...")
print("=" * 50)

result1 = asyncio.get_event_loop().run_until_complete(
    agent.ainvoke({"messages": [("user", task1)]})
)

print("AGENT RESPONSE:")
print(result1["messages"][-1].content)


---
## Step 9 — Task 2: Format Citations in APA Style

The agent reads the raw references file and formats each into proper APA citations.


In [ ]:
task2 = """
Read the file 'references.txt' in the workspace.
For each raw reference, use the format_citation tool to produce a proper APA citation.
The references are:
1. Vaswani, year 2017, title "Attention Is All You Need", journal NeurIPS
2. Devlin, year 2019, title "BERT: Pre-training of Deep Bidirectional Transformers", journal NAACL
3. Brown, year 2020, title "Language Models are Few-Shot Learners", journal NeurIPS

Write the formatted citations to a new file called 'formatted_citations.txt'.
"""

print("Running Task 2: Citation formatting...")
print("=" * 50)

result2 = asyncio.get_event_loop().run_until_complete(
    agent.ainvoke({"messages": [("user", task2)]})
)

print("AGENT RESPONSE:")
print(result2["messages"][-1].content)


---
## Step 10 — Task 3: Generate Full Literature Review Report

The agent synthesizes everything into a structured Markdown report and saves it.


In [ ]:
task3 = """
Your final task is to produce a complete literature review.

Steps:
1. Read all three paper files (paper_attention.txt, paper_bert.txt, paper_gpt.txt)
2. Extract keywords from each using extract_keywords
3. Combine all unique keywords across papers
4. Use build_summary_report to generate a Markdown report with:
   - topic: "Transformer-based Language Models"
   - papers: list the three paper titles
   - keywords: the combined unique keywords
5. Write the resulting Markdown to a file called 'literature_review.md'
6. Store the report title and date in memory under key 'last_report'
7. Confirm what was written and where.
"""

print("Running Task 3: Full literature review report...")
print("=" * 50)

result3 = asyncio.get_event_loop().run_until_complete(
    agent.ainvoke({"messages": [("user", task3)]})
)

print("AGENT RESPONSE:")
print(result3["messages"][-1].content)


---
## Step 11 — Verify Outputs


In [ ]:
from pathlib import Path

workspace = Path(WORKDIR)
print("Files in workspace after agent run:")
for f in sorted(workspace.iterdir()):
    print(f"  {f.name:40} {f.stat().st_size:>6} bytes")

# Display the generated report
report_path = workspace / "literature_review.md"
if report_path.exists():
    print("\n" + "=" * 60)
    print("GENERATED LITERATURE REVIEW:")
    print("=" * 60)
    print(report_path.read_text())
else:
    print("\nNote: literature_review.md not found in workspace.")
    print("Check the agent response above — the file may have been")
    print("written with a different name or the agent reported its content directly.")


---
## Step 12 — Inspect Agent Tool Call Trace

We replay the last run to see which tools the agent called and in what order.


In [ ]:
print("Tool calls made during Task 3 (literature review):")
print("-" * 50)
for i, msg in enumerate(result3["messages"]):
    msg_type = type(msg).__name__
    if msg_type == "AIMessage" and hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  CALL -> {tc['name']}")
            args_preview = str(tc.get("args", {}))[:80]
            print(f"    args: {args_preview}...")
    elif msg_type == "ToolMessage":
        content_preview = str(msg.content)[:80].replace("\n", " ")
        print(f"  RESULT: {content_preview}...")
print("-" * 50)
print(f"Total messages in conversation: {len(result3['messages'])}")


---
## Step 13 — Architecture Summary

```
User Query
    |
    v
Gemini 2.0 Flash (LangGraph ReAct agent)
    |
    |-- filesystem tools (read/write files in /research_workspace)
    |       @modelcontextprotocol/server-filesystem
    |
    |-- memory tools (persist key-value notes across turns)
    |       @modelcontextprotocol/server-memory
    |
    `-- research_tools (custom FastMCP server)
            - ping()                   health check
            - extract_keywords(text)   parse Keywords: line + freq analysis
            - format_citation(...)     APA formatting
            - build_summary_report()   generate Markdown report
```

### Why this design?
- **LLM decides the flow** — the agent chooses which tools to call and in what order. No hard-coded pipeline.
- **Separation of concerns** — each MCP server is independently runnable and testable.
- **Composable** — adding a new tool (e.g. `search_arxiv`, `translate_abstract`) requires only adding a `@mcp.tool` function to the custom server.
- **Persistent memory** — the memory server lets the agent remember findings across sessions.

### Extensions
- Add `@mcp.tool search_arxiv(query)` to fetch real papers via the arXiv API
- Add a `git` MCP server to track report versions
- Replace Gemini with Claude or GPT-4 by swapping the `ChatGoogleGenerativeAI` line
- Deploy the custom server as a persistent HTTP service instead of stdio subprocess
